# Logs
- Portal and Server

In [ ]:
from arcgis.gis import GIS
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import json

In [ ]:
portal_url = r"https://central.udot.utah.gov/portal/"

portal_user = r"gissiteadmin"
portal_pass = r"G1sManager"

gis = GIS(portal_url, portal_user, portal_pass)
gis

## Helper Functions

In [ ]:
def esridate(dt):
    """convert an Esri timestamp (integer) to a human readable string"""

    return datetime.fromtimestamp(dt//1000).astimezone().strftime('%a %d %b %Y %H:%M:%S (UTC%z)')

## Datetime Window definition

In [ ]:
# log end date (newest timestamp)
end_timestamp = datetime.now(tz=ZoneInfo("America/Denver"))  # define the local time zone

# set the number of previous days to query from now()
day_duration = timedelta(days=20)

# log start date (oldest timestamp)
start_timestamp = end_timestamp - day_duration

## Portal Logs

In [ ]:
log_level = [
    'OFF',
    'SEVERE',
    'WARNING',
    'INFO',
    'FINE',
    'VERBOSE',
    'DEBUG'
    ]

# remove any messages with the following codes
code_filter = [
    2180344,        # The ArcGIS Enterprise deployment has not been backed up in two weeks
    ]

In [ ]:
# get the logs as a dict
logs = gis.admin.logs.query(start_timestamp, end_timestamp, log_level[2])

# replace values in the logs dict

# convert the start and stop timestamps to human-readable strings
logs["startTime"] = esridate(logs["startTime"])
logs["endTime"] = esridate(logs["endTime"])

# filter out any codes in the code_filter list
logs['logMessages'] = [msg for msg in logs['logMessages'] if msg['code'] not in code_filter]

# change the message timestamps to a human-readable format
logs['logMessages'] = [
    {**msg, 'time': esridate(msg['time'])}
    for msg in logs['logMessages']
    ]

In [ ]:
# pretty print the results
log_json = json.dumps(logs, indent=4)
print(log_json)

In [ ]:
# Invalid sign-in attempts

# get all Users who were unsuccessful at signing in

from collections import Counter

cnt = Counter()

fails = [msg['message'].split('Username: ')[1:] for msg in logs['logMessages'] if msg['code'] == 219999]
# fails = [msg['message'] for msg in logs['logMessages'] if msg['code'] == 219999]

for username in fails:

    if username:
        # remove the various quotes and whitespace from the 'message' string
        u0 = username[0].replace('"', '').replace("'", "").strip()
        u1 = username[1].replace('"', '').replace("'", "").strip()

        if u0 == u1:
            cnt[u0] += 1
        else:
            cnt[u0] += 1
            cnt[u1] += 1
    else:
        cnt[" -- client_id not specified -- "] += 1



fails = [(cnt[x], x) for x in cnt]
fails.sort(reverse=True)


for username in fails:
    print(f"{username[1]:<40}  {username[0]:>3} attempts")


## Server Logs

In [ ]:
server_url = r"https://central.udot.utah.gov/server/admin"
